<a href="https://colab.research.google.com/github/ahsendygl/LoginSystemCsharp/blob/master/votesystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Blockchain Oylama Sistemi</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; }
        .container { background-color: #fff; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); max-width: 800px; margin: auto; }
        h1, h2 { color: #333; }
        button { padding: 10px 15px; margin: 5px; border: none; border-radius: 5px; cursor: pointer; background-color: #007bff; color: white; }
        button:hover { background-color: #0056b3; }
        input[type="text"] { padding: 10px; margin: 5px; border-radius: 5px; border: 1px solid #ddd; width: calc(100% - 22px); }
        .message { margin-top: 15px; padding: 10px; border-radius: 5px; background-color: #e2e3e5; color: #333; }
        .error { background-color: #f8d7da; color: #721c24; }
        .success { background-color: #d4edda; color: #155724; }
        #candidatesList { list-style-type: none; padding: 0; }
        #candidatesList li { background-color: #e9ecef; margin-bottom: 5px; padding: 10px; border-radius: 5px; display: flex; justify-content: space-between; align-items: center; }
    </style>
</head>
<body>
    <div class="container">
        <h1>Blockchain Oylama Sistemi</h1>
        <p>Hesap: <span id="accountAddress">Yükleniyor...</span></p>

        <div class="message" id="statusMessage">Bekliyor...</div>

        <h2>Aday Ekle</h2>
        <input type="text" id="candidateName" placeholder="Aday Adı">
        <button id="addCandidateBtn">Aday Ekle</button>

        <h2>Adaylar</h2>
        <ul id="candidatesList">
            <li>Yükleniyor...</li>
        </ul>

        <h2>Oy Ver</h2>
        <input type="text" id="voteCandidateId" placeholder="Aday ID'si">
        <button id="voteBtn">Oy Ver</button>

        <h2>Oylamayı Bitir</h2>
        <button id="endVotingBtn">Oylamayı Bitir</button>

        <h2>Kazananı Gör</h2>
        <button id="getWinnerBtn">Kazananı Göster</button>
        <p id="winnerResult"></p>
    </div>

    <script src="https://cdn.jsdelivr.net/npm/web3@latest/dist/web3.min.js"></script>
    <script src="app.js"></script>
</body>
</html>


Overwriting index.html


In [ ]:
%%writefile app.js
document.addEventListener('DOMContentLoaded', () => {
    let web3;
    let accounts;
    let contract;

    // Buraya dağıtılan akıllı sözleşmenizin adresini ve ABI'sini girin
    const CONTRACT_ADDRESS = '0xF2E246BB76DF876Cef8b38ae84130F4F55De395b'; // Örnek: '0xabc...123'
    const CONTRACT_ABI = [ {
    "inputs": [],
    "stateMutability": "nonpayable",
    "type": "constructor"
  },
  {
    "inputs": [
      {
        "internalType": "string",
        "name": "_name",
        "type": "string"
      }
    ],
    "name": "addCandidate",
    "outputs": [],
    "stateMutability": "nonpayable",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "name": "candidates",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "id",
        "type": "uint256"
      },
      {
        "internalType": "string",
        "name": "name",
        "type": "string"
      },
      {
        "internalType": "uint256",
        "name": "voteCount",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "candidatesCount",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "endVoting",
    "outputs": [],
    "stateMutability": "nonpayable",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "getWinner",
    "outputs": [
      {
        "internalType": "string",
        "name": "",
        "type": "string"
      },
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "owner",
    "outputs": [
      {
        "internalType": "address",
        "name": "",
        "type": "address"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "_candidateId",
        "type": "uint256"
      }
    ],
    "name": "vote",
    "outputs": [],
    "stateMutability": "nonpayable",
    "type": "function"
  }

    ];

    const statusMessage = document.getElementById('statusMessage');
    const accountAddress = document.getElementById('accountAddress');
    const candidateNameInput = document.getElementById('candidateName');
    const addCandidateBtn = document.getElementById('addCandidateBtn');
    const candidatesList = document.getElementById('candidatesList');
    const voteCandidateIdInput = document.getElementById('voteCandidateId');
    const voteBtn = document.getElementById('voteBtn');
    const endVotingBtn = document.getElementById('endVotingBtn');
    const getWinnerBtn = document.getElementById('getWinnerBtn');
    const winnerResult = document.getElementById('winnerResult');

    const displayMessage = (message, isError = false) => {
        statusMessage.textContent = message;
        statusMessage.className = isError ? 'message error' : 'message success';
    };

    const loadWeb3 = async () => {
        if (window.ethereum) {
            web3 = new Web3(window.ethereum);
            try {
                // Request account access if needed
                await window.ethereum.request({ method: 'eth_requestAccounts' });
                displayMessage('MetaMask bağlandı.');
            } catch (error) {
                displayMessage('MetaMask bağlantısı reddedildi.', true);
            }
        } else if (window.web3) {
            web3 = new Web3(window.web3.currentProvider);
            displayMessage('Eski bir web3 sağlayıcısı bulundu.');
        } else {
            displayMessage('Bir Web3 tarayıcısı (MetaMask gibi) algılanmadı. Lütfen yükleyin.', true);
        }
    };

    const loadBlockchainData = async () => {
        if (!web3) return;
        accounts = await web3.eth.getAccounts();
        accountAddress.textContent = accounts[0];
        contract = new web3.eth.Contract(CONTRACT_ABI, CONTRACT_ADDRESS);
        displayMessage('Sözleşme yüklendi ve veri alınıyor...');
        await renderCandidates();
    };

    const renderCandidates = async () => {
        candidatesList.innerHTML = '';
        const candidatesCount = await contract.methods.candidatesCount().call();
        if (candidatesCount == 0) {
            candidatesList.innerHTML = '<li>Henüz aday yok.</li>';
            return;
        }

        for (let i = 1; i <= candidatesCount; i++) {
            const candidate = await contract.methods.candidates(i).call();
            const listItem = document.createElement('li');
            listItem.textContent = `ID: ${candidate.id}, İsim: ${candidate.name}, Oy Sayısı: ${candidate.voteCount}`;
            candidatesList.appendChild(listItem);
        }
    };

    addCandidateBtn.addEventListener('click', async () => {
        const name = candidateNameInput.value;
        if (!name) {
            displayMessage('Lütfen bir aday adı girin.', true);
            return;
        }
        try {
            await contract.methods.addCandidate(name).send({ from: accounts[0] });
            displayMessage(`${name} adayı başarıyla eklendi.`);
            candidateNameInput.value = '';
            await renderCandidates();
        } catch (error) {
            displayMessage('Aday eklenirken hata oluştu: ' + error.message, true);
        }
    });

    voteBtn.addEventListener('click', async () => {
        const candidateId = voteCandidateIdInput.value;
        if (!candidateId || isNaN(candidateId)) {
            displayMessage('Lütfen geçerli bir aday ID'si girin.', true);
            return;
        }
        try {
            await contract.methods.vote(candidateId).send({ from: accounts[0] });
            displayMessage(`${candidateId} ID'li adaya oy verildi.`);
            voteCandidateIdInput.value = '';
            await renderCandidates();
        } catch (error) {
            displayMessage('Oy kullanırken hata oluştu: ' + error.message, true);
        }
    });

    endVotingBtn.addEventListener('click', async () => {
        try {
            await contract.methods.endVoting().send({ from: accounts[0] });
            displayMessage('Oylama başarıyla sona erdirildi.');
        } catch (error) {
            displayMessage('Oylamayı sonlandırırken hata oluştu: ' + error.message, true);
        }
    });

    getWinnerBtn.addEventListener('click', async () => {
        try {
            const result = await contract.methods.getWinner().call();
            if (result[0] === 'Beraberlik var') {
                winnerResult.textContent = `Beraberlik! En yüksek oy: ${result[1]}`;
            } else {
                winnerResult.textContent = `Kazanan: ${result[0]} (${result[1]} oy)`;
            }
            displayMessage('Kazanan başarıyla alındı.');
        } catch (error) {
            winnerResult.textContent = '';
            displayMessage('Kazanan alınırken hata oluştu: ' + error.message, true);
        }
    });

    // Initial load
    loadWeb3().then(() => {
        if (web3) {
            loadBlockchainData();
        }
    });
});

Overwriting app.js
